<h1>GROQ API SETUP</h1>

In [1]:
from secret_key import groq_api_key
import os

os.environ['GROQ_API_KEY'] = groq_api_key

In [2]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

D:\python practice\Retail Q&A Tool\retail_qa\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<h1>CONNECT WITH SQL DB</h1>

In [3]:
from langchain_community.utilities import SQLDatabase

db_user = "root"
db_password = "Prakhar%402005"  # %40 for @
db_host = "localhost"
db_name = "atliq_tshirts"

db = SQLDatabase.from_uri(f"mysql+pymysql://{db_user}:{db_password}@{db_host}/{db_name}",sample_rows_in_table_info=3)
print(db.table_info)


CREATE TABLE discounts (
	discount_id INTEGER NOT NULL AUTO_INCREMENT, 
	t_shirt_id INTEGER NOT NULL, 
	pct_discount DECIMAL(5, 2), 
	PRIMARY KEY (discount_id), 
	CONSTRAINT discounts_ibfk_1 FOREIGN KEY(t_shirt_id) REFERENCES t_shirts (t_shirt_id), 
	CONSTRAINT discounts_chk_1 CHECK ((`pct_discount` between 0 and 100))
)ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE utf8mb4_0900_ai_ci

/*
3 rows from discounts table:
discount_id	t_shirt_id	pct_discount
1	1	10.00
2	2	15.00
3	3	20.00
*/


CREATE TABLE t_shirts (
	t_shirt_id INTEGER NOT NULL AUTO_INCREMENT, 
	brand ENUM('Van Huesen','Levi','Nike','Adidas') NOT NULL, 
	color ENUM('Red','Blue','Black','White') NOT NULL, 
	size ENUM('XS','S','M','L','XL') NOT NULL, 
	price INTEGER, 
	stock_quantity INTEGER NOT NULL, 
	PRIMARY KEY (t_shirt_id), 
	CONSTRAINT t_shirts_chk_1 CHECK ((`price` between 10 and 50))
)ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE utf8mb4_0900_ai_ci

/*
3 rows from t_shirts table:
t_shirt_id	brand	color	size	price	stock

In [4]:
from langchain_core.prompts import PromptTemplate

def run_sql_from_question(llm, db, question: str):
    # Step 1: Prompt for SQL generation
    prompt = PromptTemplate.from_template("""
    You are an expert SQL generator.

    Question: {question}

    Return ONLY SQL query. No explanation, no markdown.
    """)

    # Step 2: Generate SQL query
    response = llm.invoke(prompt.format(question=question))
    sql_query = response.content

    # Clean output (in case LLM adds markdown/backticks)
    sql_query = sql_query.replace("```sql", "").replace("```", "").strip()

    # Step 3: Execute SQL and return result
    result = db.run(sql_query)

    return result

In [6]:
qns1 = run_sql_from_question(
    llm,
    db,
    "SELECT stock_quantity FROM t_shirts WHERE brand = 'Nike' AND color = 'White' AND size = 'XS'"
)

print(qns1)

[(16,)]


In [73]:
qns1

'[(16,)]'

In [9]:
qns2 = run_sql_from_question(
    llm,
    db,
    "SELECT SUM(price*stock_quantity) FROM t_shirts WHERE size = 'S'"
)

print(qns2)

[(Decimal('22676'),)]


In [10]:
sql_code = """
select sum(a.total_amount * ((100-COALESCE(discounts.pct_discount,0))/100)) as total_revenue from
(select sum(price*stock_quantity) as total_amount, t_shirt_id from t_shirts where brand = 'Levi'
group by t_shirt_id) a left join discounts on a.t_shirt_id = discounts.t_shirt_id
 """

qns3 = run_sql_from_question(
    llm,
    db,
    sql_code
)

print(qns3)

[(Decimal('27471.900000'),)]


In [11]:
qns4 = run_sql_from_question(
    llm,
    db,
    "SELECT SUM(price * stock_quantity) FROM t_shirts WHERE brand = 'Levi'"
)

print(qns4)

[(Decimal('28025'),)]


In [12]:
qns5 = run_sql_from_question(
    llm,
    db,
    "SELECT sum(stock_quantity) FROM t_shirts WHERE brand = 'Levi' AND color = 'White'"
)

print(qns5)

[(Decimal('199'),)]


**The LLM doesn't understand the names of the tables and other schemas, and also applies incorrect logic in the sql queries**<br>
**SOLUTION: FEW SHOT LEARNING**

<h1>FEW SHOT LEARNING</h1>

In [13]:
few_shots = [
    {'Question' : "How many t-shirts do we have left for Nike in XS size and white color?",
     'SQLQuery' : "SELECT sum(stock_quantity) FROM t_shirts WHERE brand = 'Nike' AND color = 'White' AND size = 'XS'",
     'SQLResult': "Result of the SQL query",
     'Answer' : qns1},
    {'Question': "How much is the total price of the inventory for all S-size t-shirts?",
     'SQLQuery':"SELECT SUM(price*stock_quantity) FROM t_shirts WHERE size = 'S'",
     'SQLResult': "Result of the SQL query",
     'Answer': qns2},
    {'Question': "If we have to sell all the Levi’s T-shirts today with discounts applied. How much revenue  our store will generate (post discounts)?" ,
     'SQLQuery' : """SELECT sum(a.total_amount * ((100-COALESCE(discounts.pct_discount,0))/100)) as total_revenue from
    (select sum(price*stock_quantity) as total_amount, t_shirt_id from t_shirts where brand = 'Levi'
    group by t_shirt_id) a left join discounts on a.t_shirt_id = discounts.t_shirt_id
     """,
     'SQLResult': "Result of the SQL query",
     'Answer': qns3} ,
     {'Question' : "If we have to sell all the Levi’s T-shirts today. How much revenue our store will generate without discount?" ,
      'SQLQuery': "SELECT SUM(price * stock_quantity) FROM t_shirts WHERE brand = 'Levi'",
      'SQLResult': "Result of the SQL query",
      'Answer' : qns4},
    {'Question': "How many white color Levi's shirt I have?",
     'SQLQuery' : "SELECT sum(stock_quantity) FROM t_shirts WHERE brand = 'Levi' AND color = 'White'",
     'SQLResult': "Result of the SQL query",
     'Answer' : qns5
     }
]

<h3>Creating Semantic Similarity Based example selector</h3>
<ul>
    <li>Create embeddings on the few shots.</li>
    <li>Store the embeddings in ChromaDB.</li>
    <li>Retrieve the the top most Semantically close example from the vector store.</li>
</ul>

In [16]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

Loading weights: 100%|█████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 6117.69it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [18]:
to_vectorise = [" ".join(example.values()) for example in few_shots]   # To create an aggregated single string to convert into vector
to_vectorise[0]

"How many t-shirts do we have left for Nike in XS size and white color? SELECT sum(stock_quantity) FROM t_shirts WHERE brand = 'Nike' AND color = 'White' AND size = 'XS' Result of the SQL query [(16,)]"

<h3>Create a ChromaDB vector database</h3>

In [20]:
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_texts(to_vectorise, embedding=embeddings, metadatas=few_shots)

<h3>Implementation of Similarity Matching</h3>

In [28]:
from langchain_core.example_selectors import SemanticSimilarityExampleSelector

example_selector = SemanticSimilarityExampleSelector(
    vectorstore = vectorstore,
    k = 2  # Pull 2 most similar examples
)

example_selector.select_examples({"Question": "How many Adidas T shirts I have left in my store?"})

[{'Answer': '[(16,)]',
  'Question': 'How many t-shirts do we have left for Nike in XS size and white color?',
  'SQLQuery': "SELECT sum(stock_quantity) FROM t_shirts WHERE brand = 'Nike' AND color = 'White' AND size = 'XS'",
  'SQLResult': 'Result of the SQL query'},
 {'SQLResult': 'Result of the SQL query',
  'Question': "How many white color Levi's shirt I have?",
  'Answer': "[(Decimal('199'),)]",
  'SQLQuery': "SELECT sum(stock_quantity) FROM t_shirts WHERE brand = 'Levi' AND color = 'White'"}]

<h3>A custom prompt to the LLM to avoid hallucination</h3>

In [30]:
### my sql based instruction prompt
mysql_prompt = """You are a MySQL expert. Given an input question, first create a syntactically correct MySQL query to run, then look at the results of the query and return the answer to the input question.
Unless the user specifies in the question a specific number of examples to obtain, query for at most {top_k} results using the LIMIT clause as per MySQL. You can order the results to return the most informative data in the database.
Never query for all columns from a table. You must query only the columns that are needed to answer the question. Wrap each column name in backticks (`) to denote them as delimited identifiers.
Pay attention to use only the column names you can see in the tables below. Be careful to not query for columns that do not exist. Also, pay attention to which column is in which table.
Pay attention to use CURDATE() function to get the current date, if the question involves "today".

Use the following format:

Question: Question here
SQLQuery: Query to run with no pre-amble
SQLResult: Result of the SQLQuery
Answer: Final answer here

No pre-amble.
"""

<h3>Setting up prompt template using input variables</h3>

In [34]:
PROMPT_SUFFIX = """Only use the following tables:
{table_info}

Question: {input}
"""

In [79]:
example_prompt = PromptTemplate(
    input_variables=["Question", "SQLQuery", "SQLResult","Answer",],
    template="\nQuestion: {Question}\nSQLQuery: {SQLQuery}\nSQLResult: {SQLResult}\nAnswer: {Answer}",
)

In [35]:
from langchain_core.prompts import FewShotPromptTemplate

few_shot_prompt = FewShotPromptTemplate(
    example_selector=example_selector,
    example_prompt=example_prompt,
    prefix=mysql_prompt,
    suffix=PROMPT_SUFFIX,
    input_variables=["input", "table_info", "top_k"], #These variables are used in the prefix and suffix
)

In [64]:
from langchain_experimental.sql import SQLDatabaseChain

new_chain = SQLDatabaseChain.from_llm(llm, db, verbose=True, prompt=few_shot_prompt, return_direct=True)

In [65]:
new_chain.invoke("How many white color Levi's shirt do I have?")



> Entering new SQLDatabaseChain chain...
How many white color Levi's shirt do I have?
SQLQuery:Question: How many white color Levi's shirt do I have?
SQLQuery: SELECT sum(`stock_quantity`) FROM `t_shirts` WHERE `brand` = 'Levi' AND `color` = 'White'
SQLResult: [(Decimal('199'),)]
> Finished chain.


{'query': "How many white color Levi's shirt do I have?",
 'result': "[(Decimal('199'),)]"}

In [66]:
new_chain.invoke("How much is the price of the inventory for all small size t-shirts?")



> Entering new SQLDatabaseChain chain...
How much is the price of the inventory for all small size t-shirts?
SQLQuery:Question: How much is the price of the inventory for all small size t-shirts?
SQLQuery: SELECT SUM(`price`*`stock_quantity`) AS `total_price` FROM `t_shirts` WHERE `size` = 'S'
SQLResult: [(Decimal('22676'),)]
> Finished chain.


{'query': 'How much is the price of the inventory for all small size t-shirts?',
 'result': "[(Decimal('22676'),)]"}

In [67]:
new_chain.invoke("How much is the price of the inventory for all extra small size t-shirts?")



> Entering new SQLDatabaseChain chain...
How much is the price of the inventory for all extra small size t-shirts?
SQLQuery:Question: How much is the price of the inventory for all extra small size t-shirts?
SQLQuery: SELECT SUM(`price`*`stock_quantity`) FROM `t_shirts` WHERE `size` = 'XS'
SQLResult: [(Decimal('13721'),)]
> Finished chain.


{'query': 'How much is the price of the inventory for all extra small size t-shirts?',
 'result': "[(Decimal('13721'),)]"}

In [68]:
new_chain.invoke("If we have to sell all the Nike’s T-shirts today with discounts applied. How much revenue  our store will generate (post discounts)?")



> Entering new SQLDatabaseChain chain...
If we have to sell all the Nike’s T-shirts today with discounts applied. How much revenue  our store will generate (post discounts)?
SQLQuery:SQLQuery: SELECT sum(a.`price` * a.`stock_quantity` * ((100-COALESCE(d.`pct_discount`,0))/100)) as total_revenue from 
    t_shirts a left join discounts d on a.`t_shirt_id` = d.`t_shirt_id` 
    where a.`brand` = 'Nike'
SQLResult: [(Decimal('17162.400000'),)]
> Finished chain.


{'query': 'If we have to sell all the Nike’s T-shirts today with discounts applied. How much revenue  our store will generate (post discounts)?',
 'result': "[(Decimal('17162.400000'),)]"}

In [69]:
qns = new_chain.invoke('How much revenue  our store will generate by selling all Van Heuson TShirts without discount?')



> Entering new SQLDatabaseChain chain...
How much revenue  our store will generate by selling all Van Heuson TShirts without discount?
SQLQuery:SQLQuery: SELECT SUM(`price` * `stock_quantity`) AS `total_revenue` FROM `t_shirts` WHERE `brand` = 'Van Huesen'
SQLResult: [(Decimal('21776'),)]
> Finished chain.


In [70]:
qns['result']

"[(Decimal('21776'),)]"

In [78]:
import ast

cleaned = qns['result'].replace("Decimal('", "").replace("')", "")
value = ast.literal_eval(cleaned)[0][0]

print(int(value))

21776
